# Forecasting Extrínseco de Series Temporales en Procesos de Moldeo por Inyección
## Cuaderno 2: Entrenamiento y Optimización de Modelos Predictivos

**Autor:** Juan Miguel Martínez Muñoz  
**Institución:** Facultad de Ingeniería de la Universidad Nacional de Asunción (FIUNA)  
**Programa:** Maestría en Ciencias de la Inteligencia Artificial  

---

## 1. Objetivo del Cuaderno
Este cuaderno toma los tensores limpios y truncados (a 3 segundos) generados en la etapa de preprocesamiento para entrenar y optimizar algoritmos predictivos. El propósito es establecer un mapeo matemático robusto entre la dinámica termomecánica temprana (*Sequence*) y las 8 variables estáticas finales de calidad (*Scalar*).

## 2. Preparación y Escalado de Datos
Antes de inyectar los datos en los modelos, se realizan las siguientes transformaciones críticas:
* **División de Conjuntos (Train/Test Split):** Separación de los datos en conjuntos de entrenamiento y prueba para garantizar una evaluación imparcial.
* **Escalado (StandardScaler):** Normalización de las características para acelerar la convergencia de las redes neuronales y estandarizar el peso de las variables en los ensambles.
  * *Nota Metodológica:* El escalador se ajusta (`fit`) **estrictamente sobre el conjunto de entrenamiento** y luego se aplica (`transform`) a los datos de prueba, bloqueando cualquier fuga de información (*Data Leakage*).

## 3. Enfoque 1: Modelos de Machine Learning (Características Estáticas)
Para establecer un *baseline* robusto, se extrajeron características estadísticas manuales (media, desviación estándar, integral) a partir del fragmento temporal de 3 segundos. Se entrenaron los siguientes algoritmos bajo una estrategia nativa *Multi-Output*:
* **Random Forest Regressor:** Ensamble de árboles de decisión altamente resistente al sobreajuste. Sus hiperparámetros fueron optimizados mediante búsqueda exhaustiva (`GridSearchCV`).
* **XGBoost (Extreme Gradient Boosting):** Algoritmo de boosting diseñado para capturar relaciones no lineales complejas entre la termodinámica inicial y los escalares resultantes.

## 4. Enfoque 2: Modelos de Deep Learning (Representaciones Secuenciales)
Se utilizaron los tensores tridimensionales (secuencias crudas truncadas) para aprovechar la capacidad de las redes neuronales profundas en la extracción automática de características espaciales y temporales:
* **CNN-LSTM:** Arquitectura híbrida. Las capas Convolucionales 1D extraen patrones espaciales locales del flujo de presión, mientras que la capa LSTM modela la dependencia temporal secuencial.
* **FCN (Fully Convolutional Network):** Red puramente convolucional adaptada para series temporales 1D. Se encarga de mapear representaciones latentes a través de bloques residuales antes de proyectarlas a las 8 salidas escalares.

## 5. Exportación
Los modelos entrenados, junto con los objetos de escalado, se serializan y exportan para su posterior inferencia y validación en la fase de evaluación.

In [2]:
import os
import random
import math
import numpy as np
import torch
import pandas as pd
from google.colab import drive
from sklearn.model_selection import train_test_split
from sklearn.multioutput import MultiOutputRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error, r2_score
from sklearn.model_selection import GridSearchCV
from xgboost import XGBRegressor
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
import joblib
import time

def set_seed(seed=42):
    """
    Fija todas las semillas de aleatoriedad para garantizar reproducibilidad.
    """
    # 1. Python core
    random.seed(seed)
    # Bloquear el hash de Python (afecta a diccionarios y sets)
    os.environ['PYTHONHASHSEED'] = str(seed)

    # 2. NumPy
    np.random.seed(seed)

    # 3. PyTorch (CPU y GPU)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed) # en caso de usar multiples gpus

        # Forzar operaciones deterministas en cuDNN
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(42)
print("Semilla global fijada en 42 para reproducibilidad.")

Semilla global fijada en 42 para reproducibilidad.


In [3]:
# ==============================================================================
# CELDA 1: Configuración de Colab, Drive y Extracción de Características
# ==============================================================================

print("--- 1. MONTANDO GOOGLE DRIVE ---")
# Esto permitirá guardar/cargar checkpoints más adelante
drive.mount('/content/drive')

# ruta al csv (procedente del preprocesamiento)
ruta_csv = '/content/drive/MyDrive/TP_Final_SeriesTemporales/datos_inyeccion_preprocesados.csv'
df = pd.read_csv(ruta_csv)

print("\n--- 2. EXTRACCIÓN DE CARACTERÍSTICAS ESTADÍSTICAS ---")
features_cols = ['MEAS_pressure_frontsensor_bar', 'MEAS_pressure_backsensor_bar', 'MEAS_pressure_difference_bar']

# NUEVA LISTA DE 8 TARGETS
targets_cols = [
    'MEAS_KM_Zyklus- Zeit Z [s]',
    'MEAS_KM_Plast- Zeit Z [s]',
    'MEAS_KM_Umsch MasDrZ [bar]',
    'MEAS_KM_Max MasDr [bar]',
    'MEAS_KM_Fläche Massedr [bar*s]',
    'MEAS_KM_Max F Leistung [kW]',
    'MEAS_KM_Max StauDr [bar]',
    'MEAS_KM_Energie PeriAuto [Wh]'
]

ciclos = list(df.groupby(['META_experiment', 'META_run']))
X_estadistico = []
y_objetivos = []

for _, datos_ciclo in ciclos:
    fila_features = []
    # Extraemos estadísticas descriptivas por cada sensor
    for sensor in features_cols:
        serie = datos_ciclo[sensor].values
        fila_features.extend([
            np.mean(serie),
            np.std(serie),
            np.max(serie),
            np.min(serie),
            np.trapezoid(serie) # Área bajo la curva (Integral)
        ])
    X_estadistico.append(fila_features)

    # Los targets son constantes por ciclo, tomamos el primero
    y_objetivos.append(datos_ciclo[targets_cols].iloc[0].values)

X = np.array(X_estadistico)
y = np.array(y_objetivos)

# PRINTS DINÁMICOS CORREGIDOS
print(f"Dimensiones de X (Tabular): {X.shape} -> {X.shape[0]} muestras, {X.shape[1]} características (5 por sensor)")
print(f"Dimensiones de Y (Targets): {y.shape} -> {y.shape[0]} muestras, {y.shape[1]} objetivos")

# División de datos (80% entrenamiento, 20% prueba)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("\n--- 3. ESCALADO DE CARACTERÍSTICAS PREDICTORAS ---")
# Instanciamos el escalador
scaler_X = StandardScaler()

# FIT y TRANSFORM solo en el set de entrenamiento
X_train_scaled = scaler_X.fit_transform(X_train)

# SOLO TRANSFORM en el set de prueba (usando los parámetros aprendidos del train)
X_test_scaled = scaler_X.transform(X_test)

print(f"X_train_scaled shape: {X_train_scaled.shape}")
print(f"X_test_scaled shape: {X_test_scaled.shape}")
print("Nota: Las variables objetivo (Y) se mantienen en su escala original.")

--- 1. MONTANDO GOOGLE DRIVE ---
Mounted at /content/drive

--- 2. EXTRACCIÓN DE CARACTERÍSTICAS ESTADÍSTICAS ---
Dimensiones de X (Tabular): (60, 15) -> 60 muestras, 15 características (5 por sensor)
Dimensiones de Y (Targets): (60, 8) -> 60 muestras, 8 objetivos

--- 3. ESCALADO DE CARACTERÍSTICAS PREDICTORAS ---
X_train_scaled shape: (48, 15)
X_test_scaled shape: (12, 15)
Nota: Las variables objetivo (Y) se mantienen en su escala original.


In [4]:
# ==============================================================================
# CELDA 2: Entrenamiento y Optimización de Modelos ML (GridSearch)
# ==============================================================================

print("--- 3. BÚSQUEDA DE HIPERPARÁMETROS Y ENTRENAMIENTO (ML) ---")

# ---------------------------------------------------------
# Modelo 1: Random Forest
# ---------------------------------------------------------
print("\nIniciando GridSearchCV para Random Forest...")
start_time = time.time()

rf_base = RandomForestRegressor(random_state=42)
param_grid_rf = {
    'n_estimators': [100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5]
}

grid_rf = GridSearchCV(estimator=rf_base, param_grid=param_grid_rf,
                       cv=3, n_jobs=-1, scoring='neg_mean_squared_error', verbose=1)
# CAMBIO: Usamos X_train_scaled
grid_rf.fit(X_train_scaled, y_train)

best_rf = grid_rf.best_estimator_
print(f"Mejores parámetros RF: {grid_rf.best_params_}")
print(f"Tiempo RF: {(time.time() - start_time):.2f} segundos")

# ---------------------------------------------------------
# Modelo 2: XGBoost
# ---------------------------------------------------------
print("\nIniciando GridSearchCV para XGBoost...")
start_time = time.time()

# Nota: Al usar MultiOutputRegressor, los parámetros de XGBoost deben llevar el prefijo 'estimator__'
xgb_base = MultiOutputRegressor(XGBRegressor(random_state=42))
param_grid_xgb = {
    'estimator__n_estimators': [100, 200],
    'estimator__max_depth': [3, 5, 7],
    'estimator__learning_rate': [0.05, 0.1]
}

grid_xgb = GridSearchCV(estimator=xgb_base, param_grid=param_grid_xgb,
                        cv=3, n_jobs=-1, scoring='neg_mean_squared_error', verbose=1)
# CAMBIO: Usamos X_train_scaled
grid_xgb.fit(X_train_scaled, y_train)

best_xgb = grid_xgb.best_estimator_
print(f"Mejores parámetros XGB: {grid_xgb.best_params_}")
print(f"Tiempo XGB: {(time.time() - start_time):.2f} segundos")

--- 3. BÚSQUEDA DE HIPERPARÁMETROS Y ENTRENAMIENTO (ML) ---

Iniciando GridSearchCV para Random Forest...
Fitting 3 folds for each of 12 candidates, totalling 36 fits
Mejores parámetros RF: {'max_depth': None, 'min_samples_split': 2, 'n_estimators': 200}
Tiempo RF: 11.49 segundos

Iniciando GridSearchCV para XGBoost...
Fitting 3 folds for each of 12 candidates, totalling 36 fits
Mejores parámetros XGB: {'estimator__learning_rate': 0.05, 'estimator__max_depth': 3, 'estimator__n_estimators': 200}
Tiempo XGB: 9.79 segundos


In [5]:
# ==============================================================================
# CELDA 3: Guardado de Modelos Optimizados y Escalador
# ==============================================================================
import joblib

# Guardamos los MEJORES modelos extraídos del GridSearchCV
joblib.dump(best_rf, 'best_rf_model.joblib')
joblib.dump(best_xgb, 'best_xgb_model.joblib')

# CAMBIO: Guardamos scaler_X en lugar de my_scaler
# Esto servirá para escalar los datos de prueba en la notebook de Evaluación
joblib.dump(scaler_X, 'scaler_X.joblib')

print("Modelos optimizados y escalador guardados correctamente para inferencia.")

Modelos optimizados y escalador guardados correctamente para inferencia.


In [6]:
# ==============================================================================
# CELDA 4: Evaluación de los Modelos Clásicos Optimizados (Métricas)
# ==============================================================================
import math
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error, r2_score

print("--- 4. EVALUACIÓN DE MÉTRICAS MULTI-OUTPUT (MODELOS OPTIMIZADOS) ---")

# CAMBIO: Generamos predicciones usando X_test_scaled
y_pred_rf = best_rf.predict(X_test_scaled)
y_pred_xgb = best_xgb.predict(X_test_scaled)

# Definimos una función para imprimir el reporte limpio
def evaluar_modelo(nombre_modelo, y_true, y_pred, columnas_objetivo):
    print(f"\nResultados para {nombre_modelo}:")
    print("-" * 50)

    # Iteramos sobre cada una de las 8 variables de salida
    for i, target in enumerate(columnas_objetivo):
        y_t = y_true[:, i]
        y_p = y_pred[:, i]

        mae = mean_absolute_error(y_t, y_p)
        rmse = math.sqrt(mean_squared_error(y_t, y_p))
        mape = mean_absolute_percentage_error(y_t, y_p) * 100
        r2 = r2_score(y_t, y_p)

        print(f"Variable: {target}")
        print(f"  * MAE:  {mae:.4f}")
        print(f"  * RMSE: {rmse:.4f}")
        print(f"  * MAPE: {mape:.2f}%")
        print(f"  * R²:   {r2:.4f}\n")

# Ejecutamos el reporte para los modelos optimizados
evaluar_modelo("Random Forest (Optimizado - GridSearchCV)", y_test, y_pred_rf, targets_cols)
evaluar_modelo("XGBoost (Optimizado - GridSearchCV)", y_test, y_pred_xgb, targets_cols)

--- 4. EVALUACIÓN DE MÉTRICAS MULTI-OUTPUT (MODELOS OPTIMIZADOS) ---

Resultados para Random Forest (Optimizado - GridSearchCV):
--------------------------------------------------
Variable: MEAS_KM_Zyklus- Zeit Z [s]
  * MAE:  0.0475
  * RMSE: 0.0613
  * MAPE: 0.11%
  * R²:   0.9820

Variable: MEAS_KM_Plast- Zeit Z [s]
  * MAE:  0.0565
  * RMSE: 0.0725
  * MAPE: 0.43%
  * R²:   0.9803

Variable: MEAS_KM_Umsch MasDrZ [bar]
  * MAE:  9.7283
  * RMSE: 13.0510
  * MAPE: 1.16%
  * R²:   0.9883

Variable: MEAS_KM_Max MasDr [bar]
  * MAE:  6.9650
  * RMSE: 8.0629
  * MAPE: 0.72%
  * R²:   0.9967

Variable: MEAS_KM_Fläche Massedr [bar*s]
  * MAE:  33.6163
  * RMSE: 46.3541
  * MAPE: 0.29%
  * R²:   0.9790

Variable: MEAS_KM_Max F Leistung [kW]
  * MAE:  0.2522
  * RMSE: 0.2993
  * MAPE: 4.10%
  * R²:   0.9851

Variable: MEAS_KM_Max StauDr [bar]
  * MAE:  1.3721
  * RMSE: 1.8281
  * MAPE: 0.24%
  * R²:   0.3202

Variable: MEAS_KM_Energie PeriAuto [Wh]
  * MAE:  0.1284
  * RMSE: 0.2316
  * MAPE:

In [7]:
# ==============================================================================
# CELDA: Refactorización de Dataset (Sin Fuga de Datos) y Loaders
# ==============================================================================


# 1. Definición de la clase refactorizada
class InjectionDataset(Dataset):
    def __init__(self, list_of_cycles, targets_cols, features_cols, scaler_y=None, scaler_X=None, is_train=True):
        self.cycles = list_of_cycles
        self.targets_cols = targets_cols
        self.features_cols = features_cols
        self.is_train = is_train

        # --- A. Procesamiento de Targets (Y) ---
        targets_raw = np.array([cycle_data[self.targets_cols].iloc[0].values for _, cycle_data in self.cycles])
        if self.is_train:
            self.scaler_y = StandardScaler()
            self.targets_scaled = self.scaler_y.fit_transform(targets_raw)
        else:
            self.scaler_y = scaler_y
            self.targets_scaled = self.scaler_y.transform(targets_raw)

        # --- B. Procesamiento de Features (X) ---
        # Extraemos todas las series temporales en un solo tensor grande para calcular media/std global
        X_raw_list = [cycle_data[self.features_cols].values for _, cycle_data in self.cycles]
        # X_raw_list tiene forma [num_ciclos, timesteps, num_sensores]

        if self.is_train:
            self.scaler_X = StandardScaler()
            # Aplanamos temporalmente para ajustar el escalador a todos los puntos de datos
            X_flat = np.vstack(X_raw_list)
            self.scaler_X.fit(X_flat)

        else:
            self.scaler_X = scaler_X

        # Aplicamos la transformación ciclo por ciclo y transponemos a [canales, longitud]
        self.X_scaled = []
        for x_seq in X_raw_list:
            x_seq_scaled = self.scaler_X.transform(x_seq)
            # Transponemos para PyTorch Conv1d: (timesteps, canales) -> (canales, timesteps)
            self.X_scaled.append(np.transpose(x_seq_scaled))

    def __len__(self):
        return len(self.cycles)

    def __getitem__(self, idx):
        X = torch.tensor(self.X_scaled[idx], dtype=torch.float32)
        y = torch.tensor(self.targets_scaled[idx], dtype=torch.float32)
        return X, y

# 2. Preparación de los datos
csv_path = '/content/drive/MyDrive/TP_Final_SeriesTemporales/datos_inyeccion_preprocesados.csv'
df = pd.read_csv(csv_path)

features_cols = ['MEAS_pressure_frontsensor_bar', 'MEAS_pressure_backsensor_bar', 'MEAS_pressure_difference_bar']
targets_cols = [
    'MEAS_KM_Zyklus- Zeit Z [s]', 'MEAS_KM_Plast- Zeit Z [s]', 'MEAS_KM_Umsch MasDrZ [bar]',
    'MEAS_KM_Max MasDr [bar]', 'MEAS_KM_Fläche Massedr [bar*s]', 'MEAS_KM_Max F Leistung [kW]',
    'MEAS_KM_Max StauDr [bar]', 'MEAS_KM_Energie PeriAuto [Wh]'
]

todos_los_ciclos = list(df.groupby(['META_experiment', 'META_run']))
train_cycles, test_cycles = train_test_split(todos_los_ciclos, test_size=0.2, random_state=42)

# 3. Instanciamos Datasets
train_dataset = InjectionDataset(train_cycles, targets_cols, features_cols, is_train=True)
# Pasamos ambos escaladores (X e Y) al set de prueba
test_dataset = InjectionDataset(
    test_cycles, targets_cols, features_cols,
    scaler_y=train_dataset.scaler_y,
    scaler_X=train_dataset.scaler_X,
    is_train=False
)

# 4. Creamos los DataLoaders
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)

print(f"Ciclos de entrenamiento: {len(train_dataset)}")
print(f"Ciclos de prueba: {len(test_dataset)}")
print("¡Preparación completada! Predictores (X) y Objetivos (Y) escalados sin fuga de datos temporal.")

Ciclos de entrenamiento: 48
Ciclos de prueba: 12
¡Preparación completada! Predictores (X) y Objetivos (Y) escalados sin fuga de datos temporal.


In [8]:
import joblib

# Guardamos el escalador de los targets (necesario para invertir predicciones en la evaluación)
joblib.dump(train_dataset.scaler_y, 'scaler_y_dl.joblib')

# Guardamos el escalador de las features (necesario por si quieres predecir sobre datos nuevos)
joblib.dump(train_dataset.scaler_X, 'scaler_X_dl.joblib')

print("Escaladores de Deep Learning guardados correctamente.")

Escaladores de Deep Learning guardados correctamente.


In [9]:
# ==============================================================================
# CELDA: Bucle de Entrenamiento y Evaluación
# ==============================================================================


def train_model(model, train_loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0

    for inputs, targets in train_loader:
        inputs = inputs.to(device)
        targets = targets.to(device)

        # Reiniciar gradientes
        optimizer.zero_grad()

        # Forward pass
        outputs = model(inputs)
        loss = criterion(outputs, targets)

        # Backward pass y optimización
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)

    epoch_loss = running_loss / len(train_loader.dataset)
    return epoch_loss

def evaluate_model(model, test_loader, criterion, device):
    model.eval()
    running_loss = 0.0

    all_targets = []
    all_predictions = []

    with torch.no_grad():
        for inputs, targets in test_loader:
            inputs = inputs.to(device)
            targets = targets.to(device)

            outputs = model(inputs)
            loss = criterion(outputs, targets)

            running_loss += loss.item() * inputs.size(0)

            # Almacenar predicciones y valores reales para calcular R2
            all_targets.append(targets.cpu().numpy())
            all_predictions.append(outputs.cpu().numpy())

    epoch_loss = running_loss / len(test_loader.dataset)

    # Concatenar para calcular métricas globales
    all_targets = np.vstack(all_targets)
    all_predictions = np.vstack(all_predictions)

    # Calcular R^2 por cada variable de salida
    r2_scores = r2_score(all_targets, all_predictions, multioutput='raw_values')

    return epoch_loss, r2_scores

In [10]:
# ==============================================================================
# CELDA: Definición de la Arquitectura CNN-LSTM Multi-Output
# ==============================================================================

class CNN_LSTM_Model(nn.Module):
    def __init__(self, input_channels=3, num_targets=8):
        super(CNN_LSTM_Model, self).__init__()

        # 1. BLOQUE CNN (Extracción de características y reducción de dimensionalidad)
        # Recibe: (Batch, 3 características, 30001 timesteps)
        self.cnn = nn.Sequential(
            nn.Conv1d(in_channels=input_channels, out_channels=16, kernel_size=5, stride=2, padding=2),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=4, stride=4),

            nn.Conv1d(in_channels=16, out_channels=32, kernel_size=5, stride=2, padding=2),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=4, stride=4),

            nn.Conv1d(in_channels=32, out_channels=64, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2, stride=2)
            # La CNN comprime los 30,001 timesteps a una secuencia manejable de aprox. 117 pasos
        )

        # 2. BLOQUE LSTM (Análisis de secuencia temporal a largo plazo)
        # input_size=64 coincide con los out_channels de la última capa CNN
        self.lstm = nn.LSTM(input_size=64, hidden_size=64, num_layers=2, batch_first=True, dropout=0.2)

        # 3. BLOQUE FULLY CONNECTED (Mapeo final a las 3 variables objetivo)
        self.fc = nn.Sequential(
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, num_targets)
        )

    def forward(self, x):
        # PyTorch espera tensores Conv1d en formato: (batch_size, canales, longitud_secuencia)
        # Si el DataLoader entrega (batch, seq, channels), lo reordenamos:
        if x.shape[2] == 3:
            x = x.permute(0, 2, 1)

        # Paso 1: Reducción con CNN
        c_out = self.cnn(x)

        # Paso 2: Reordenar para la LSTM (batch_size, longitud_secuencia, características)
        r_in = c_out.permute(0, 2, 1)

        # Paso 3: Análisis secuencial temporal
        lstm_out, (h_n, c_n) = self.lstm(r_in)

        # Paso 4: Extraer el último estado oculto (el resumen de toda la secuencia)
        last_hidden = lstm_out[:, -1, :]

        # Paso 5: Predicción final de los 3 valores (Presión, Energía, Tiempo)
        predictions = self.fc(last_hidden)

        return predictions

print("Arquitectura CNN-LSTM compilada correctamente.")

Arquitectura CNN-LSTM compilada correctamente.


In [11]:
# ==============================================================================
# CELDA: Ejecución del Entrenamiento (CNN-LSTM)
# ==============================================================================
# 1. Configuración del dispositivo (GPU si está disponible)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Entrenando en: {device}")

# 2. Instanciar el modelo
# NOTA: Reemplaza 'CNN_LSTM_Model()' con el nombre exacto de la clase de tu modelo
# asumiendo que ya tiene las dimensiones de entrada (3 canales) y salida (3 targets)
model = CNN_LSTM_Model().to(device)

# 3. Definir función de pérdida y optimizador
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)

# 4. Parámetros del ciclo
num_epochs = 50
best_val_loss = float('inf')

print("Iniciando entrenamiento...")
for epoch in range(num_epochs):
    # Entrenar
    train_loss = train_model(model, train_loader, criterion, optimizer, device)

    # Evaluar
    val_loss, val_r2 = evaluate_model(model, test_loader, criterion, device)

    # Guardar el mejor modelo
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'best_cnn_lstm_model.pth')
        saved_flag = "*"
    else:
        saved_flag = ""

    # Imprimir progreso cada 5 épocas
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch [{epoch+1:03d}/{num_epochs}] | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} {saved_flag}")
        # Mostrar array completo redondeado a 4 decimales
        print(f"          R2 Scores: {np.round(val_r2, 4)}")

print("¡Entrenamiento finalizado! Mejor modelo guardado en 'best_cnn_lstm_model.pth'")

Entrenando en: cuda
Iniciando entrenamiento...
Epoch [001/50] | Train Loss: 1.0134 | Val Loss: 1.3304 *
          R2 Scores: [-0.0915 -0.0531 -0.0181 -0.3159 -0.053  -0.1296 -0.0047 -0.2305]
Epoch [005/50] | Train Loss: 0.9964 | Val Loss: 1.3126 *
          R2 Scores: [-0.069  -0.0449 -0.0102 -0.2693 -0.0383 -0.1343  0.0043 -0.2093]
Epoch [010/50] | Train Loss: 0.7401 | Val Loss: 1.0799 
          R2 Scores: [-0.0634  0.3032  0.2384  0.0832  0.0174  0.0783  0.1952  0.0331]
Epoch [015/50] | Train Loss: 0.5729 | Val Loss: 0.9081 
          R2 Scores: [-0.0779  0.6335  0.5474  0.865   0.0457 -0.0353  0.3096 -0.0938]
Epoch [020/50] | Train Loss: 0.5248 | Val Loss: 0.8889 
          R2 Scores: [-0.0478  0.7138  0.5738  0.8861  0.0523  0.0207  0.3031 -0.1552]
Epoch [025/50] | Train Loss: 0.4827 | Val Loss: 0.7277 *
          R2 Scores: [-0.0145  0.904   0.7936  0.8916  0.1657 -0.1754  0.4417  0.2024]
Epoch [030/50] | Train Loss: 0.4776 | Val Loss: 0.7300 
          R2 Scores: [ 0.0447  0.913

In [12]:
# ==============================================================================
# CELDA: Definición de la Arquitectura FCN (Fully Convolutional Network)
# ==============================================================================

class FCN_Model(nn.Module):
    def __init__(self, input_channels=3, num_targets=8):
        super(FCN_Model, self).__init__()

        # Bloque 1: Extracción inicial
        self.conv1 = nn.Conv1d(in_channels=input_channels, out_channels=32, kernel_size=7, stride=2, padding=3)
        self.bn1 = nn.BatchNorm1d(32)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool1d(kernel_size=4, stride=4)

        # Bloque 2: Extracción de nivel medio
        self.conv2 = nn.Conv1d(in_channels=32, out_channels=64, kernel_size=5, stride=2, padding=2)
        self.bn2 = nn.BatchNorm1d(64)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool1d(kernel_size=4, stride=4)

        # Bloque 3: Extracción profunda
        self.conv3 = nn.Conv1d(in_channels=64, out_channels=128, kernel_size=3, stride=2, padding=1)
        self.bn3 = nn.BatchNorm1d(128)
        self.relu3 = nn.ReLU()

        # Global Average Pooling (GAP)
        # Promedia las características a lo largo de todo el tiempo, evitando el overfitting
        self.gap = nn.AdaptiveAvgPool1d(1)

        # Capa de salida totalmente conectada
        self.fc = nn.Linear(128, num_targets)

    def forward(self, x):
        # Asegurar el formato (batch_size, canales, longitud_secuencia)
        if x.shape[2] == 3:
            x = x.permute(0, 2, 1)

        # Paso a través de los bloques convolucionales
        x = self.pool1(self.relu1(self.bn1(self.conv1(x))))
        x = self.pool2(self.relu2(self.bn2(self.conv2(x))))
        x = self.relu3(self.bn3(self.conv3(x)))

        # Aplicar Global Average Pooling
        x = self.gap(x)

        # Aplanar el tensor de (batch, 128, 1) a (batch, 128)
        x = x.squeeze(-1)

        # Predicción final
        predictions = self.fc(x)
        return predictions

print("Arquitectura FCN compilada correctamente.")

Arquitectura FCN compilada correctamente.


In [13]:
# ==============================================================================
# CELDA: Ejecución del Entrenamiento (FCN)
# ==============================================================================
# 1. Instanciar el nuevo modelo FCN
# Nos aseguramos de mandarlo a la GPU (device ya está definido de la sesión anterior)
fcn_model = FCN_Model().to(device)

# 2. Definir función de pérdida y un nuevo optimizador para esta red
criterion = nn.MSELoss()
optimizer_fcn = optim.Adam(fcn_model.parameters(), lr=0.001, weight_decay=1e-5)

# 3. Parámetros del ciclo
num_epochs = 50
best_val_loss_fcn = float('inf')

print("Iniciando entrenamiento de la FCN...")
for epoch in range(num_epochs):
    # Entrenar
    train_loss = train_model(fcn_model, train_loader, criterion, optimizer_fcn, device)

    # Evaluar
    val_loss, val_r2 = evaluate_model(fcn_model, test_loader, criterion, device)

    # Guardar el mejor modelo
    if val_loss < best_val_loss_fcn:
        best_val_loss_fcn = val_loss
        torch.save(fcn_model.state_dict(), 'best_fcn_model.pth')
        saved_flag = "*"
    else:
        saved_flag = ""

    # Imprimir progreso cada 5 épocas
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch [{epoch+1:03d}/{num_epochs}] | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} {saved_flag}")
        # Mostrar array completo redondeado a 4 decimales
        print(f"          R2 Scores: {np.round(val_r2, 4)}")
print("¡Entrenamiento FCN finalizado! Mejor modelo guardado en 'best_fcn_model.pth'")

Iniciando entrenamiento de la FCN...
Epoch [001/50] | Train Loss: 0.9040 | Val Loss: 1.3229 *
          R2 Scores: [-0.0408 -0.2117 -0.0204 -0.1819 -0.0017 -0.2172 -0.0402 -0.1169]
Epoch [005/50] | Train Loss: 0.4828 | Val Loss: 1.0067 *
          R2 Scores: [ 0.0742  0.3454  0.3031  0.6009  0.1015  0.2447  0.069  -0.0715]
Epoch [010/50] | Train Loss: 0.4187 | Val Loss: 0.7945 
          R2 Scores: [ 0.3991  0.6657  0.4791  0.7792  0.3037  0.3857  0.3376 -0.3884]
Epoch [015/50] | Train Loss: 0.4120 | Val Loss: 0.6366 *
          R2 Scores: [0.436  0.6336 0.5931 0.5736 0.3953 0.4268 0.3986 0.4351]
Epoch [020/50] | Train Loss: 0.3270 | Val Loss: 0.5444 
          R2 Scores: [0.4335 0.8745 0.7902 0.9212 0.4646 0.3292 0.4711 0.2293]
Epoch [025/50] | Train Loss: 0.3531 | Val Loss: 0.4389 *
          R2 Scores: [0.6887 0.8795 0.8174 0.9953 0.6425 0.711  0.4378 0.1675]
Epoch [030/50] | Train Loss: 0.3448 | Val Loss: 0.4815 
          R2 Scores: [0.6802 0.8894 0.7769 0.9926 0.5974 0.6546 0.420

In [14]:
# evaluacion celda... deeplearning... mejor version....

In [15]:
# Imprimir comparativa extendida
def print_extended_comparison(metrics1, name1, metrics2, name2):
    print(f"================ COMPARATIVA COMPLETA (DL) ================")

    # ACTUALIZADO A LAS 8 VARIABLES EXACTAS
    variables = [
        'Tiempo de Ciclo',
        'Tiempo de Plastificacion',
        'Presion Conmutacion',
        'Presion Max Masa',
        'Integral Presion',
        'Potencia Max',
        'Presion Remanso',
        'Energia Consumida'
    ]

    for var in variables:
        print(f"\n--- {var} ---")
        data = {
            "Métrica": ["MAE", "RMSE", "MAPE (%)", "R²"],
            name1: [f"{metrics1[var]['MAE']:.4f}", f"{metrics1[var]['RMSE']:.4f}", f"{metrics1[var]['MAPE']:.2f}%", f"{metrics1[var]['R2']:.4f}"],
            name2: [f"{metrics2[var]['MAE']:.4f}", f"{metrics2[var]['RMSE']:.4f}", f"{metrics2[var]['MAPE']:.2f}%", f"{metrics2[var]['R2']:.4f}"]
        }
        print(pd.DataFrame(data).to_string(index=False))

In [16]:
# ==============================================================================
# CELDA: Evaluación comparativa rigurosa (Forzando carga de mejores pesos)
# ==============================================================================

# Accedemos al escalador que se ajustó (fit) durante el entrenamiento
my_scaler = train_dataset.scaler_y

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error

def evaluate_rigorous(model, model_path, test_loader, device, scaler):
    # 1. FORZAMOS la carga del mejor modelo guardado
    model.load_state_dict(torch.load(model_path))
    model.to(device)
    model.eval()

    all_targets = []
    all_predictions = []

    with torch.no_grad():
        for inputs, targets in test_loader:
            inputs = inputs.to(device)
            outputs = model(inputs)
            all_targets.append(targets.cpu().numpy())
            all_predictions.append(outputs.cpu().numpy())

    all_targets = np.vstack(all_targets)
    all_predictions = np.vstack(all_predictions)

    # Invertir el escalado
    real_targets = scaler.inverse_transform(all_targets)
    real_predictions = scaler.inverse_transform(all_predictions)

    metrics = {}
    variables = [
        'Tiempo de Ciclo',
        'Tiempo de Plastificacion',
        'Presion Conmutacion',
        'Presion Max Masa',
        'Integral Presion',
        'Potencia Max',
        'Presion Remanso',
        'Energia Consumida'
    ]

    for i, var in enumerate(variables):
        mae = mean_absolute_error(real_targets[:, i], real_predictions[:, i])
        rmse = np.sqrt(mean_squared_error(real_targets[:, i], real_predictions[:, i]))
        mape = mean_absolute_percentage_error(real_targets[:, i], real_predictions[:, i]) * 100
        r2 = r2_score(real_targets[:, i], real_predictions[:, i])
        metrics[var] = {'MAE': mae, 'RMSE': rmse, 'MAPE': mape, 'R2': r2}

    return metrics

# Re-evaluamos usando la lógica rigurosa
metrics_hybrid_full = evaluate_rigorous(CNN_LSTM_Model().to(device), 'best_cnn_lstm_model.pth', test_loader, device, my_scaler)
metrics_fcn_full = evaluate_rigorous(FCN_Model().to(device), 'best_fcn_model.pth', test_loader, device, my_scaler)

# Imprimir comparativa
# (Usa la misma función print_extended_comparison de antes)
print_extended_comparison(metrics_hybrid_full, "CNN-LSTM", metrics_fcn_full, "FCN")

================ COMPARATIVA COMPLETA (DL) ================

--- Tiempo de Ciclo ---
 Métrica CNN-LSTM    FCN
     MAE   0.1163 0.1620
    RMSE   0.1399 0.1841
MAPE (%)    0.28%  0.39%
      R²   0.9061 0.8374

--- Tiempo de Plastificacion ---
 Métrica CNN-LSTM    FCN
     MAE   0.1303 0.1158
    RMSE   0.1711 0.1341
MAPE (%)    0.98%  0.88%
      R²   0.8904 0.9326

--- Presion Conmutacion ---
 Métrica CNN-LSTM     FCN
     MAE  10.9141 29.7574
    RMSE  15.2542 36.7413
MAPE (%)    1.42%   3.63%
      R²   0.9840  0.9072

--- Presion Max Masa ---
 Métrica CNN-LSTM     FCN
     MAE  48.8004 14.0038
    RMSE  60.6961 16.4526
MAPE (%)    4.61%   1.41%
      R²   0.8104  0.9861

--- Integral Presion ---
 Métrica CNN-LSTM      FCN
     MAE  71.9034 113.7423
    RMSE  85.7539 145.4305
MAPE (%)    0.63%    1.00%
      R²   0.9280   0.7929

--- Potencia Max ---
 Métrica CNN-LSTM    FCN
     MAE   0.8368 0.9448
    RMSE   1.1927 1.1020
MAPE (%)   13.90% 14.31%
      R²   0.7634 0.7980

--- Pre